In [1]:
import re
import sys
import numpy as np
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [2]:
!ls ../data/raw

HomeCredit_columns_description.csv bureau_balance.csv
POS_CASH_balance.csv               credit_card_balance.csv
applications.csv                   installments_payments.csv
bureau.csv                         previous_application.csv


In [3]:
raw_dir_candidates = [Path("../data/raw"), Path("data/raw")]
RAW_DIR = next((path for path in raw_dir_candidates if path.is_dir()), None)


### Tables Explorations

In [4]:
TABLES = {
    "applications": {
        "file": "applications.csv",
        "description": (
            "The main table. Contains static data for applications; "
            "one row represents one loan in the sample."
        ),
    },
    "bureau": {
        "file": "bureau.csv",
        "description": (
            "Clients' previous credits from other financial institutions, as reported "
            "to the Credit Bureau. A client can have multiple rows, one for each "
            "previous credit."
        ),
    },
    "bureau_balance": {
        "file": "bureau_balance.csv",
        "description": (
            "Monthly balances for previous credits reported to the Credit Bureau. "
            "Each row represents one month of history for one previous credit."
        ),
    },
    "POS_CASH_balance": {
        "file": "POS_CASH_balance.csv",
        "description": (
            "Monthly balance snapshots for previous POS and cash loans with "
            "Home Credit. Each row represents one month of history for one previous loan."
        ),
    },
    "credit_card_balance": {
        "file": "credit_card_balance.csv",
        "description": (
            "Monthly balance snapshots for previous Home Credit credit cards. "
            "Each row represents one month of history for one previous credit card."
        ),
    },
    "previous_application": {
        "file": "previous_application.csv",
        "description": (
            "Clients' previous applications for Home Credit loans. "
            "Each row represents one previous application related to a loan in the sample."
        ),
    },
    "installments_payments": {
        "file": "installments_payments.csv",
        "description": (
            "Repayment history for previously disbursed Home Credit loans. "
            "Rows represent payments made or missed installments."
        ),
    },
    "HomeCredit_columns_description": {
        "file": "HomeCredit_columns_description.csv",
        "description": (
            "Descriptions of columns used across the Home Credit data files."
        ),
    }
}
missing_files = [
    info["file"]
    for info in TABLES.values()
    if not (RAW_DIR / info["file"]).exists()
]
if missing_files:
    raise FileNotFoundError(f"Missing files in {RAW_DIR}: {missing_files}")

def read_csv_safely(path):
    # The column-description file may use a non-UTF-8 encoding.
    try:
        return pd.read_csv(path, low_memory=False, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin-1")



In [5]:
# Full DataFrames are stored here, keyed by table name.
dfs = {}

for table_name, info in TABLES.items():
    df = read_csv_safely(RAW_DIR / info["file"])
    dfs[table_name] = df

    display(Markdown(f"## `{info['file']}`"))
    display(Markdown(info["description"]))

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")

    print("\nHead:")
    display(df.head())

    null_counts = df.isna().sum()
    metadata = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": len(df) - null_counts,
        "missing": null_counts,
        "missing_pct": (null_counts / len(df) * 100).round(2),
    })

    print("Column metadata:")
    display(metadata)

    numeric_df = df.select_dtypes(include="number")
    if not numeric_df.empty:
        print("Numeric summary:")
        display(numeric_df.describe().T)

## `applications.csv`

The main table. Contains static data for applications; one row represents one loan in the sample.

Shape: 307,511 rows × 122 columns
Memory: 325.22 MB

Head:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

Column metadata:


...

,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN


Column metadata:


,dtype,non_null,missing,missing_pct
Unnamed: 0,int64,219,0,0.00
Table,str,219,0,0.00
Row,str,219,0,0.00
Description,str,219,0,0.00
Special,str,86,133,60.73


Numeric summary:


,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,219.0,111.981735,63.395726,1.0,57.5,112.0,166.5,221.0


In [6]:
dfs['HomeCredit_columns_description']


,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
5,8,application_{train|test}.csv,FLAG_OWN_REALTY,Flag if client owns a house or flat,NaN
6,9,application_{train|test}.csv,CNT_CHILDREN,Number of children the client has,NaN
7,10,application_{train|test}.csv,AMT_INCOME_TOTAL,Income of the client,NaN
8,11,application_{train|test}.csv,AMT_CREDIT,Credit amount of the loan,NaN
9,12,application_{train|test}.csv,AMT_ANNUITY,Loan annuity,NaN


# Applications

### Funcitons:

In [7]:
def load_applications(path=None):
    """Load applications.csv from the project root or a notebook subfolder."""
    if path is None:
        candidates = [
            Path("../data/raw/applications.csv"),
            Path("data/raw/applications.csv"),
        ]
        path = next((p for p in candidates if p.is_file()), None)

    if path is None or not Path(path).is_file():
        raise FileNotFoundError("Could not find applications.csv in data/raw.")

    return pd.read_csv(path, low_memory=False)


def check_key_duplicates(df, key="SK_ID_CURR"):
    """Report missing and duplicate key values; does not remove any rows."""
    if key not in df.columns:
        raise KeyError(f"Missing key column: {key}")

    missing_count = int(df[key].isna().sum())
    duplicate_mask = df[key].notna() & df[key].duplicated(keep=False)
    duplicate_ids = df.loc[duplicate_mask, key].nunique()
    extra_rows = int(df.loc[df[key].notna(), key].duplicated().sum())

    print(f"{key} missing values: {missing_count:,}")
    print(f"Duplicate {key} values: {duplicate_ids:,}")
    print(f"Rows beyond the first occurrence of a duplicated key: {extra_rows:,}")

    if duplicate_mask.any():
        print("Example rows with duplicated keys:")
        display(df.loc[duplicate_mask].sort_values(key).head(10))

    return {
        "missing_keys": missing_count,
        "duplicate_key_values": int(duplicate_ids),
        "extra_duplicate_rows": extra_rows,
    }


def summarize_categories(df, columns=None, max_values=20):
    """Show value counts for TARGET, categorical columns, and FLAG_* columns."""
    if columns is None:
        categorical = df.select_dtypes(
            include=["object", "category", "bool", "string"]
        ).columns.tolist()
        flags = [col for col in df.columns if col.startswith("FLAG_")]
        columns = ["TARGET"] + categorical + flags

    # Keep only existing columns and remove repeats while preserving order.
    columns = list(dict.fromkeys(col for col in columns if col in df.columns))

    for col in columns:
        counts = df[col].value_counts(dropna=False)
        show_all = col == "TARGET" or col.startswith("FLAG_")
        shown = counts if show_all else counts.head(max_values)

        summary = pd.DataFrame({
            "count": shown,
            "percent": (shown / len(df) * 100).round(2),
        })

        print(f"\n{col} — {df[col].nunique(dropna=False)} unique values")
        display(summary)

        if not show_all and len(counts) > max_values:
            print(f"Showing the top {max_values} values.")


def summarize_numeric_distributions(df, exclude=("SK_ID_CURR", "TARGET")):
    """Return descriptive statistics and missingness for numeric columns."""
    columns = [
        col for col in df.select_dtypes(include="number").columns
        if col not in exclude
    ]

    if not columns:
        print("No numeric feature columns found.")
        return pd.DataFrame()

    result = df[columns].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T

    result["missing"] = df[columns].isna().sum()
    result["missing_pct"] = (df[columns].isna().mean() * 100).round(2)

    display(result)
    return result
def add_document_flag_sum(df, new_column="NUM_DOCUMENTS"):
    document_columns = [
        col for col in df.columns
        if re.fullmatch(r"FLAG_DOCUMENT_\d+", col)
    ]
    if not document_columns:
        raise ValueError("No FLAG_DOCUMENT_<number> columns found.")

    values = df[document_columns].to_numpy(
        dtype=np.float32,
        na_value=np.nan,
        copy=False,
    )

    if np.isnan(values).any():
        sums = np.nansum(values, axis=1)
        sums[np.isnan(values).all(axis=1)] = np.nan
    else:
        sums = values.sum(axis=1)

    df[new_column] = sums
    df.drop(columns=document_columns, inplace=True)
    return df


def inspect_and_convert_mode_columns(df, value_maps=None):
    """Inspect string/object columns ending in _MODE and safely convert supported values."""
    result = df.copy()

    if value_maps is None:
        value_maps = {
            "EMERGENCYSTATE_MODE": {"yes": 1.0, "no": 0.0}
        }

    mode_columns = [
        col for col in result.columns
        if col.endswith("_MODE")
        and (
            pd.api.types.is_object_dtype(result[col])
            or pd.api.types.is_string_dtype(result[col])
            or isinstance(result[col].dtype, pd.CategoricalDtype)
        )
    ]

    for col in mode_columns:
        print(f"\n{col} | dtype before: {result[col].dtype}")
        display(result[col].value_counts(dropna=False))

        values = result[col].astype("string").str.strip()

        # Convert numeric strings, such as "0" or "1".
        numeric_values = pd.to_numeric(values, errors="coerce")
        if values.dropna().eq(numeric_values.dropna().astype("string")).all():
            result[col] = numeric_values.astype("float64")
            print(f"Converted {col} to float64.")
            continue

        # Convert categories only when an explicit mapping covers every observed value.
        mapping = value_maps.get(col)
        if mapping is not None:
            normalized_mapping = {
                str(key).strip().casefold(): value
                for key, value in mapping.items()
            }
            normalized_values = values.str.casefold()
            unknown = normalized_values.dropna().loc[
                ~normalized_values.dropna().isin(normalized_mapping)
            ].unique()

            if len(unknown) == 0:
                result[col] = normalized_values.map(normalized_mapping).astype("float64")
                print(f"Converted {col} to float64 using its value mapping.")
                continue

            print(f"Skipped {col}; unmapped values found: {unknown}")

        else:
            print(f"Skipped {col}; define a value mapping before converting categories.")

    return result

### Cleaning and Feature Engineering

In [8]:
applications = dfs['applications']
check_key_duplicates(applications)


SK_ID_CURR missing values: 0
Duplicate SK_ID_CURR values: 0
Rows beyond the first occurrence of a duplicated key: 0


{'missing_keys': 0, 'duplicate_key_values': 0, 'extra_duplicate_rows': 0}

In [9]:
summarize_categories(applications)       # Includes TARGET and FLAG_* values


TARGET — 2 unique values


,count,percent
TARGET,,
0,282686,91.93
1,24825,8.07



NAME_CONTRACT_TYPE — 2 unique values


,count,percent
NAME_CONTRACT_TYPE,,
Cash loans,278232,90.48
Revolving loans,29279,9.52



CODE_GENDER — 3 unique values


...

,count,percent
FLAG_DOCUMENT_19,,
0,307328,99.94
1,183,0.06



FLAG_DOCUMENT_20 — 2 unique values


,count,percent
FLAG_DOCUMENT_20,,
0,307355,99.95
1,156,0.05



FLAG_DOCUMENT_21 — 2 unique values


,count,percent
FLAG_DOCUMENT_21,,
0,307408,99.97
1,103,0.03


In [10]:
def inspect_flag_dtypes(df):
    """Show dtypes and observed values for flag and related columns."""
    columns = [
        col for col in df.columns
        if col.startswith("FLAG_")
        or col in ["EMERGENCYSTATE_MODE", "CODE_GENDER"]
    ]

    for col in columns:
        values = df[col].dropna().unique()
        print(f"{col}: dtype={df[col].dtype}, values={values[:20]}")

inspect_flag_dtypes(applications)


CODE_GENDER: dtype=str, values=<ArrowStringArray>
['M', 'F', 'XNA']
Length: 3, dtype: str
FLAG_OWN_CAR: dtype=str, values=<ArrowStringArray>
['N', 'Y']
Length: 2, dtype: str
FLAG_OWN_REALTY: dtype=str, values=<ArrowStringArray>
['Y', 'N']
Length: 2, dtype: str
FLAG_MOBIL: dtype=int64, values=[1 0]
FLAG_EMP_PHONE: dtype=int64, values=[1 0]
FLAG_WORK_PHONE: dtype=int64, values=[0 1]
FLAG_CONT_MOBILE: dtype=int64, values=[1 0]
FLAG_PHONE: dtype=int64, values=[1 0]
FLAG_EMAIL: dtype=int64, values=[0 1]
EMERGENCYSTATE_MODE: dtype=str, values=<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
FLAG_DOCUMENT_2: dtype=int64, values=[0 1]
FLAG_DOCUMENT_3: dtype=int64, values=[1 0]
FLAG_DOCUMENT_4: dtype=int64, values=[0 1]
FLAG_DOCUMENT_5: dtype=int64, values=[0 1]
FLAG_DOCUMENT_6: dtype=int64, values=[0 1]
FLAG_DOCUMENT_7: dtype=int64, values=[0 1]
FLAG_DOCUMENT_8: dtype=int64, values=[0 1]
FLAG_DOCUMENT_9: dtype=int64, values=[0 1]
FLAG_DOCUMENT_10: dtype=int64, values=[0 1]
FLAG_DOCUMENT_

In [11]:

def convert_application_dtypes(df):
    """Convert specified categorical fields, preserving missing values."""
    result = df.copy()

    # Ownership flags: Y -> 1, N -> 0
    for col in ["FLAG_OWN_REALTY", "FLAG_OWN_CAR"]:
        if col in result.columns:
            values = result[col].astype("string").str.strip().str.upper()
            result[col] = values.map({"Y": 1, "N": 0}).astype("Int8")

    # Emergency-state flag: Yes -> 1, No -> 0
    col = "EMERGENCYSTATE_MODE"
    if col in result.columns:
        values = result[col].astype("string").str.strip().str.lower()
        result[col] = values.map({"yes": 1, "no": 0}).astype("Int8")

    # Invalid gender code XNA -> missing; retain M and F
    col = "CODE_GENDER"
    if col in result.columns:
        values = result[col].astype("string").str.strip().str.upper()
        result[col] = values.replace("XNA", pd.NA)

    return result
inter_df = convert_application_dtypes(applications)


In [12]:
numeric_summary = summarize_numeric_distributions(inter_df)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing,missing_pct
FLAG_OWN_CAR,307511.0,0.340108,0.473746,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0,0.00
FLAG_OWN_REALTY,307511.0,0.693673,0.460968,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0,0.00
CNT_CHILDREN,307511.0,0.417052,0.722121,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,19.0,0,0.00
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,45000.0,67500.0,112500.0,147150.0,202500.0,337500.0,472500.0,117000000.0,0,0.00
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,76410.0,135000.0,270000.0,513531.0,808650.0,1350000.0,1854000.0,4050000.0,0,0.00
AMT_ANNUITY,307499.0,27108.573909,14493.737315,1615.5,6182.91,9000.0,16524.0,24903.0,34596.0,53325.0,70006.5,258025.5,12,0.00
AMT_GOODS_PRICE,307233.0,538396.207429,369446.46054,40500.0,67500.0,135000.0,238500.0,450000.0,679500.0,1305000.0,1800000.0,4050000.0,278,0.09
REGION_POPULATION_RELATIVE,307511.0,0.020868,0.013831,0.00029,0.002134,0.00496,0.010006,0.01885,0.028663,0.04622,0.072508,0.072508,0,0.00
DAYS_BIRTH,307511.0,-16036.995067,4363.988632,-25229.0,-24419.0,-23204.0,-19682.0,-15750.0,-12413.0,-9407.0,-8263.0,-7489.0,0,0.00
DAYS_EMPLOYED,307511.0,63815.045904,141275.766519,-17912.0,-10894.9,-6742.5,-2760.0,-1213.0,-289.0,365243.0,365243.0,365243.0,0,0.00


In [13]:
inter_df = add_document_flag_sum(inter_df)


/var/folders/sk/3z7vkbcj6m3640z6_9d52g040000gq/T/ipykernel_51340/3174134991.py:111: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=document_columns, inplace=True)


In [14]:
inter_df[["SK_ID_CURR", "NUM_DOCUMENTS"]].head()

,SK_ID_CURR,NUM_DOCUMENTS
0,100002,1.0
1,100003,1.0
2,100004,0.0
3,100006,1.0
4,100007,1.0


# Splitting & Final DataFrame Saving

In [15]:
from pathlib import Path
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    inter_df,
    test_size=0.20,
    random_state=42,
    stratify=inter_df["TARGET"],
)

output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(output_dir / "train.parquet", index=False)
test_df.to_parquet(output_dir / "test.parquet", index=False)